In [ ]:
# === Environment Detection ===
# Detect which platform this notebook is running on.
# Works across Colab, Kaggle, Datalore, DeepNote, and local.
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
from urllib.request import urlopen
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
IS_DATALORE = any(k.startswith("DATALORE") for k in os.environ) or os.path.exists("/data/notebook_files")
IS_DEEPNOTE = any(k.startswith("DEEPNOTE") for k in os.environ) or os.path.exists("/work")
IS_CLOUD = IS_COLAB or IS_KAGGLE or IS_DATALORE or IS_DEEPNOTE

# === Parse pinned versions from pyproject.toml ===
# Fetch the repo's pyproject.toml and extract dependency versions.
# This keeps the notebook in sync with the repo automatically.
PYPROJECT_URL = "https://raw.githubusercontent.com/dramirezbe/notebook-toolkit-gcpds/main/pyproject.toml"

toml = urlopen(PYPROJECT_URL).read().decode()
deps = []
in_deps = False
for line in toml.splitlines():
    if line.strip().startswith("dependencies"):
        in_deps = True
        continue
    if in_deps:
        if line.strip().startswith("]"):
            break
        dep = line.strip().strip(",").strip('"')
        if dep:
            deps.append(dep)

TARGET = {d.split("==")[0]: d.split("==")[1] for d in deps if "==" in d}

# === Version Check ===
# Check if the already-installed versions match the pinned targets.
# Uses importlib.metadata to avoid importing C extensions directly.
def check_versions():
    for pkg, expected in TARGET.items():
        try:
            if version(pkg) != expected:
                return False
        except PackageNotFoundError:
            return False
    return True

in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)

if check_versions():
    print("Packages already installed.")
else:
    # === Install Packages ===
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    if in_venv or IS_CLOUD:
        # Cloud or existing venv: install directly into current environment.
        # --system targets system Python (for cloud kernels).
        # --force-reinstall replaces pre-installed packages (Kaggle fix).
        uv_args = [sys.executable, "-m", "uv", "pip", "install"]
        if not in_venv:
            uv_args += ["--system", "--force-reinstall"]
        if IS_DATALORE:
            uv_args.append("--break-system-packages")
        subprocess.run([
            *uv_args,
            "notebook-toolkit-gcpds @ git+https://github.com/dramirezbe/notebook-toolkit-gcpds.git",
            *deps,
        ], check=True)

        # Auto-restart kernel to load newly installed packages.
        env = "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Datalore" if IS_DATALORE else "DeepNote" if IS_DEEPNOTE else "Local"
        print(f"Environment: {env} | Python: {sys.version.split()[0]}")
        print("Packages installed. Restarting kernel...")
        try:
            import IPython
            IPython.Application.instance().kernel.do_shutdown(restart=True)
        except Exception:
            print("Restart the kernel manually: Runtime > Restart session")
    else:
        # Bare system Python: create a local .venv and register a Jupyter kernel.
        # The user must switch to the new kernel after this cell runs.
        venv_dir = Path(".venv")
        if not venv_dir.exists():
            subprocess.run([sys.executable, "-m", "venv", ".venv"], check=True)
            venv_python = str(venv_dir / "bin" / "python")
            subprocess.run([venv_python, "-m", "pip", "install", "-q", "uv"], check=True)
            subprocess.run([
                venv_python, "-m", "uv", "pip", "install",
                "notebook-toolkit-gcpds @ git+https://github.com/dramirezbe/notebook-toolkit-gcpds.git",
                *deps,
            ], check=True)
            subprocess.run([
                venv_python, "-m", "ipykernel", "install",
                "--name", "notebook-toolkit",
                "--display-name", "Python (notebook-toolkit)",
            ], check=True)
        print("Local venv created. Switch kernel: Kernel > Change kernel > Python (notebook-toolkit)")

In [ ]:
# === Verify Installation ===
# Import and print versions to confirm the correct packages are loaded.
import numpy
import pandas
import scipy
import sklearn

print(f"NumPy:       {numpy.__version__}")
print(f"Pandas:      {pandas.__version__}")
print(f"SciPy:       {scipy.__version__}")
print(f"Scikit-Learn: {sklearn.__version__}")